# Setup

In [4]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.models import alexnet
from tqdm.auto import tqdm
import kagglehub

# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================================
# SAVE DIR (NEW NOTEBOOK SAFE)
# =========================================================

SAVE_DIR = "./saved_models_2"
os.makedirs(SAVE_DIR, exist_ok=True)

Device: cuda


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")

train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")

# =========================================================
# IMPROVED AUGMENTATION
# =========================================================

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(64, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25)
])

transform_val = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

full_dataset = datasets.ImageFolder(train_path, transform=transform_train)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

num_classes = 200

In [6]:
def train_model(model, train_loader, val_loader, epochs, optimizer, scheduler, criterion, save_name):

    scaler = torch.amp.GradScaler("cuda")
    best_acc = 0

    model.to(device)

    for epoch in range(epochs):
        model.train()
        start = time.time()

        correct = 0
        total = 0

        bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for x, y in bar:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            bar.set_postfix(loss=loss.item(), acc=100*correct/total)

        if scheduler:
            scheduler.step()

        # VALIDATION
        model.eval()
        vc, vt = 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                preds = out.argmax(1)

                vt += y.size(0)
                vc += (preds == y).sum().item()

        acc = 100 * vc / vt

        print(f"\nVal Acc: {acc:.2f}% | Time: {time.time()-start:.1f}s")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, save_name))
            print("Saved best model")

    return best_acc

# Model Definitions

In [28]:
class SmallResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.GELU(),
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c)
        )

    def forward(self, x):
        return nn.GELU()(x + self.block(x))


class ImprovedTinyCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            SmallResidualBlock(64),

            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU(),
            SmallResidualBlock(128),

            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.GELU(),
            SmallResidualBlock(256),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [29]:
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.shortcut = nn.Identity()

        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)


class StrongCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layers = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256)
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        return self.classifier(x)

In [30]:
class AlexNet3x3(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 192, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(192, 384, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((6,6))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256*6*6, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Training

In [8]:
EPOCHS = 30

In [8]:
model = ImprovedTinyCNN().to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

acc = train_model(
    model,
    train_loader,
    val_loader,
    EPOCHS,
    optimizer,
    scheduler,
    criterion,
    "improved_tiny_cnn_2.pth"
)

print("ImprovedTinyCNN:", acc)

Epoch 1/30: 100%|██████████| 1407/1407 [00:51<00:00, 27.27it/s, acc=2.99, loss=4.71]



Val Acc: 6.60% | Time: 55.0s
Saved best model


Epoch 2/30: 100%|██████████| 1407/1407 [00:51<00:00, 27.45it/s, acc=7.63, loss=4.64]



Val Acc: 9.78% | Time: 54.6s
Saved best model


Epoch 3/30: 100%|██████████| 1407/1407 [00:51<00:00, 27.38it/s, acc=11, loss=4.37]  


...


Val Acc: 34.81% | Time: 54.5s


Epoch 29/30: 100%|██████████| 1407/1407 [00:51<00:00, 27.48it/s, acc=34.3, loss=3.2] 



Val Acc: 35.23% | Time: 54.6s
Saved best model


Epoch 30/30: 100%|██████████| 1407/1407 [00:51<00:00, 27.49it/s, acc=34.7, loss=3.31]



Val Acc: 34.88% | Time: 54.5s
ImprovedTinyCNN: 35.23


In [9]:
model = StrongCNN().to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

acc = train_model(
    model,
    train_loader,
    val_loader,
    EPOCHS,
    optimizer,
    scheduler,
    criterion,
    "strong_cnn_2.pth"
)

print("StrongCNN:", acc)

Epoch 1/30: 100%|██████████| 1407/1407 [01:22<00:00, 16.96it/s, acc=2.53, loss=5.14]



Val Acc: 4.78% | Time: 88.3s
Saved best model


Epoch 2/30: 100%|██████████| 1407/1407 [01:22<00:00, 16.98it/s, acc=6.05, loss=4.76]



Val Acc: 8.17% | Time: 88.2s
Saved best model


Epoch 3/30: 100%|██████████| 1407/1407 [01:22<00:00, 16.96it/s, acc=9.5, loss=4.4]  


...


Val Acc: 37.61% | Time: 88.2s
Saved best model


Epoch 29/30: 100%|██████████| 1407/1407 [01:22<00:00, 16.96it/s, acc=37.3, loss=3.08]



Val Acc: 37.77% | Time: 88.3s
Saved best model


Epoch 30/30: 100%|██████████| 1407/1407 [01:22<00:00, 16.97it/s, acc=37.3, loss=3.41]



Val Acc: 37.44% | Time: 88.3s
StrongCNN: 37.77


In [10]:
model = alexnet(weights="IMAGENET1K_V1")
model.classifier[6] = nn.Linear(4096, 200)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

acc = train_model(
    model,
    train_loader,
    val_loader,
    EPOCHS,
    optimizer,
    scheduler,
    criterion,
    "alexnet_2.pth"
)

print("AlexNet:", acc)

Epoch 1/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.37it/s, acc=11.3, loss=4.46]



Val Acc: 15.61% | Time: 48.6s
Saved best model


Epoch 2/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.46it/s, acc=17, loss=4.51]  



Val Acc: 17.77% | Time: 48.6s
Saved best model


Epoch 3/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.46it/s, acc=19.5, loss=4.18]


...


Val Acc: 28.98% | Time: 48.5s
Saved best model


Epoch 29/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.44it/s, acc=37.1, loss=3.45]



Val Acc: 28.21% | Time: 48.5s


Epoch 30/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.45it/s, acc=37.6, loss=2.97]



Val Acc: 28.83% | Time: 48.5s
AlexNet: 28.98


In [ ]:
model = AlexNet3x3().to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

acc = train_model(
    model,
    train_loader,
    val_loader,
    EPOCHS,
    optimizer,
    scheduler,
    criterion,
    "alexnet3x3_2.pth"
)

print("AlexNet3x3:", acc)

Epoch 1/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.58it/s, acc=1.45, loss=4.98]



Val Acc: 2.83% | Time: 48.3s
Saved best model


Epoch 2/30: 100%|██████████| 1407/1407 [00:45<00:00, 30.62it/s, acc=3.8, loss=5.02] 



Val Acc: 4.70% | Time: 48.2s
Saved best model


Epoch 3/30: 100%|██████████| 1407/1407 [00:46<00:00, 30.59it/s, acc=5.99, loss=4.8] 


...

Epoch 27/30: 100%|██████████| 1407/1407 [00:49<00:00, 28.70it/s, acc=28.2, loss=3.32]



Val Acc: 25.42% | Time: 53.7s
Saved best model


Epoch 28/30: 100%|██████████| 1407/1407 [00:49<00:00, 28.70it/s, acc=28.3, loss=3.4] 



Val Acc: 25.71% | Time: 53.7s
Saved best model


Epoch 29/30:  45%|████▌     | 637/1407 [1:38:43<00:26, 28.68it/s, acc=28.2, loss=3.63]

# Hopefully Really Good Model

In [38]:
class SEBlock(nn.Module):

    def __init__(self, channels, reduction=4):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.GELU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):

        b, c, _, _ = x.shape

        y = self.pool(x).view(b, c)

        y = self.fc(y).view(b, c, 1, 1)

        return x * y

In [39]:
class ConvNeXtBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.dwconv = nn.Conv2d(
            dim,
            dim,
            kernel_size=7,
            padding=3,
            groups=dim
        )

        self.norm = nn.BatchNorm2d(dim)

        self.pwconv1 = nn.Conv2d(
            dim,
            4 * dim,
            kernel_size=1
        )

        self.act = nn.GELU()

        self.pwconv2 = nn.Conv2d(
            4 * dim,
            dim,
            kernel_size=1
        )

        self.se = SEBlock(dim)

        self.gamma = nn.Parameter(
            1e-6 * torch.ones(dim)
        )

    def forward(self, x):

        residual = x

        x = self.dwconv(x)

        x = self.norm(x)

        x = self.pwconv1(x)

        x = self.act(x)

        x = self.pwconv2(x)

        x = self.se(x)

        x = self.gamma.view(
            1,
            -1,
            1,
            1
        ) * x

        return residual + x

In [40]:
class ConvNeXtSETiny(nn.Module):

    def __init__(self, num_classes=200):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(
                3,
                96,
                kernel_size=4,
                stride=4
            ),

            nn.BatchNorm2d(96)
        )

        self.stage1 = nn.Sequential(
            ConvNeXtBlock(96),
            ConvNeXtBlock(96),
            ConvNeXtBlock(96)
        )

        self.down1 = nn.Sequential(
            nn.BatchNorm2d(96),
            nn.Conv2d(
                96,
                192,
                kernel_size=2,
                stride=2
            )
        )

        self.stage2 = nn.Sequential(
            ConvNeXtBlock(192),
            ConvNeXtBlock(192),
            ConvNeXtBlock(192)
        )

        self.down2 = nn.Sequential(
            nn.BatchNorm2d(192),
            nn.Conv2d(
                192,
                384,
                kernel_size=2,
                stride=2
            )
        )

        self.stage3 = nn.Sequential(
            ConvNeXtBlock(384),
            ConvNeXtBlock(384),
            ConvNeXtBlock(384),
            ConvNeXtBlock(384),
            ConvNeXtBlock(384),
            ConvNeXtBlock(384)
        )

        self.down3 = nn.Sequential(
            nn.BatchNorm2d(384),
            nn.Conv2d(
                384,
                768,
                kernel_size=2,
                stride=2
            )
        )

        self.stage4 = nn.Sequential(
            ConvNeXtBlock(768),
            ConvNeXtBlock(768),
            ConvNeXtBlock(768)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(768),
            nn.Linear(
                768,
                num_classes
            )
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)

        x = self.down1(x)

        x = self.stage2(x)

        x = self.down2(x)

        x = self.stage3(x)

        x = self.down3(x)

        x = self.stage4(x)

        x = self.pool(x)

        return self.head(x)

In [41]:
model = ConvNeXtSETiny(
    num_classes=200
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=4e-4,
    weight_decay=0.05
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [14]:
acc = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    save_name="convnext_se_tiny.pth"
)

Epoch 1/100: 100%|██████████| 1407/1407 [01:09<00:00, 20.15it/s, acc=2.91, loss=4.92]



Val Acc: 4.45% | Time: 74.4s
Saved best model


Epoch 2/100: 100%|██████████| 1407/1407 [01:00<00:00, 23.21it/s, acc=6.08, loss=4.86]



Val Acc: 7.38% | Time: 63.3s
Saved best model


Epoch 3/100: 100%|██████████| 1407/1407 [01:00<00:00, 23.23it/s, acc=8.97, loss=4.8] 


...


Val Acc: 41.79% | Time: 63.9s
Saved best model


Epoch 99/100: 100%|██████████| 1407/1407 [01:00<00:00, 23.40it/s, acc=97.6, loss=1.25] 



Val Acc: 41.35% | Time: 63.0s


Epoch 100/100: 100%|██████████| 1407/1407 [01:00<00:00, 23.38it/s, acc=97.7, loss=1.08] 



Val Acc: 41.01% | Time: 63.1s


# Ranking

In [31]:
import torch
import torch.nn as nn
from torchvision.models import alexnet
import os

# =========================================================
# EVAL FUNCTION (consistent with your setup)
# =========================================================

def evaluate_model(model, loader, device):
    model = model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            preds = out.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total

In [32]:
SAVE_DIR = "./saved_models_2"

models = {}
results = {}

In [33]:
models["ImprovedTinyCNN"] = ImprovedTinyCNN().to(device)
models["ImprovedTinyCNN"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "improved_tiny_cnn_2.pth"), map_location=device)
)

/tmp/ipykernel_63439/991554159.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "improved_tiny_cnn_2.pth"), map_location=device)


<All keys matched successfully>

In [34]:
models["StrongCNN"] = StrongCNN().to(device)
models["StrongCNN"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "strong_cnn_2.pth"), map_location=device)
)

/tmp/ipykernel_63439/2772403113.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "strong_cnn_2.pth"), map_location=device)


<All keys matched successfully>

In [35]:
alexnet_model = alexnet(weights=None)
alexnet_model.classifier[6] = nn.Linear(4096, 200)

alexnet_model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "alexnet_2.pth"), map_location=device)
)

models["AlexNet"] = alexnet_model.to(device)

/tmp/ipykernel_63439/2569694005.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "alexnet_2.pth"), map_location=device)


In [36]:
models["AlexNet3x3"] = AlexNet3x3(num_classes=200).to(device)
models["AlexNet3x3"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "alexnet3x3_2.pth"), map_location=device)
)

/tmp/ipykernel_63439/2712261149.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "alexnet3x3_2.pth"), map_location=device)


<All keys matched successfully>

In [42]:
models["ConvNeXtSETiny"] = ConvNeXtSETiny(num_classes=200).to(device)
models["ConvNeXtSETiny"].load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "convnext_se_tiny.pth"), map_location=device)
)

/tmp/ipykernel_63439/69242692.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(SAVE_DIR, "convnext_se_tiny.pth"), map_location=device)


<All keys matched successfully>

In [43]:
print("\nEVALUATING MODELS")
print("-" * 60)

for name, model in models.items():
    acc = evaluate_model(model, val_loader, device)
    results[name] = acc
    print(f"{name:20s}: {acc:.2f}%")


EVALUATING MODELS
------------------------------------------------------------
ImprovedTinyCNN     : 37.97%
StrongCNN           : 41.24%
AlexNet             : 38.69%
AlexNet3x3          : 30.29%
ConvNeXtSETiny      : 92.14%


In [44]:
print("\nFINAL RANKING")
print("=" * 60)

sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

for i, (name, acc) in enumerate(sorted_results, 1):
    print(f"{i:02d}. {name:20s} -> {acc:.2f}%")


FINAL RANKING
01. ConvNeXtSETiny       -> 92.14%
02. StrongCNN            -> 41.24%
03. AlexNet              -> 38.69%
04. ImprovedTinyCNN      -> 37.97%
05. AlexNet3x3           -> 30.29%
